In [232]:
import pandas as pd
import numpy as np
import optimization as opt
import matplotlib.pyplot as plt
import geopandas as gpd
import pydeck as pdk

pdk.settings.rendering = "iframe"

import optimization_plots
import importlib

importlib.reload(optimization_plots)
importlib.reload(opt)


ES = ['cpool', 'shannon', 'harvest', 'litter']
SSPs = ['ssp126', 'ssp585']
LPJ_MANAGEMENT_OPTIONS = ['spbau_manbau', 'spbau_shortrot', 'spbau_longrot', 'spbau_stop', 'tobd_manbau', 'tobd_shortrot', 'tobd_longrot', 'tone_manbau', 'tone_shortrot', 'tone_longrot']

import colorsys

base_colors = {
    "spbau": (0/255, 0/255, 255/255),
    "tobd": (0/255, 128/255, 0/255),
    "tone": (255/255, 165/255, 0/255),
    "spbau_stop": (128/255, 0/255, 128/255)
}

# (value_factor, saturation_factor)
intensity = {
    "longrot": (1.0, 0.5),   # lighter → less saturated
    "manbau": (1.0, 1.0),    # base
    "shortrot": (0.7, 1.0)   # darker → lower value
}

def adjust_color(rgb, v_factor, s_factor):
    h, s, v = colorsys.rgb_to_hsv(*rgb)

    v = min(v * v_factor, 1.0)
    s = min(s * s_factor, 1.0)

    r, g, b = colorsys.hsv_to_rgb(h, s, v)
    return (int(r * 255), int(g * 255), int(b * 255), 220)

def get_color(name):
    if name == "spbau_stop":
        r, g, b = base_colors["spbau_stop"]
        return (int(r*255), int(g*255), int(b*255), 220)

    system, management = name.split("_")
    base = base_colors[system]
    v_factor, s_factor = intensity[management]

    return adjust_color(base, v_factor, s_factor)

MANAGEMENT_COLORS = [get_color(m) for m in LPJ_MANAGEMENT_OPTIONS]


In [201]:
data_for_optimizer_pre = pd.read_csv('lpjguess_optimizer_data_unnormalized.csv', sep=';')

In [202]:
# data_for_optimizer_pre2 = data_for_optimizer_pre.set_index(['Lon', 'Lat']).loc[(-8.75, 43.25)].reset_index().set_index(['Lon', 'Lat', 'ES', 'ssp'])
# data_for_optimizer_pre3 = data_for_optimizer_pre.set_index(['Lon', 'Lat']).loc[(-8.75, 43.25)].reset_index().set_index(['Lon', 'Lat', 'ssp', 'ES'])
data_for_optimizer_pre2 = data_for_optimizer_pre.set_index(['Lon', 'Lat', 'ES', 'ssp'])

In [203]:
importlib.reload(opt)
data_for_optimizer = data_for_optimizer_pre2.groupby(['Lon', 'Lat'], group_keys=False).apply(lambda grp: opt.prepare_for_optimization(grp, ['Lon', 'Lat'], 'ssp'))

In [204]:
# TODO should not be necessary, fix this in
data_for_optimizer = data_for_optimizer.reset_index().rename(columns={'ssp': 'ES'}).rename(columns={'management': 'ssp'}).set_index(['Lon', 'Lat', 'ES', 'ssp'])

In [205]:
data_for_optimizer

spbau_manbau  spbau_shortrot  spbau_longrot  \
Lon      Lat      ES      ssp                                                   
-8.75000 42.75000 cpool   ssp126       0.00013         0.00013        0.00013   
                          ssp585       0.00010         0.00010        0.00010   
                  harvest ssp126       0.00000         0.00000        0.00000   
                          ssp585       0.00000         0.00000        0.00000   
                  litter  ssp126       0.00000         0.00000        0.00000   
...                                        ...             ...            ...   
24.75000 61.75000 harvest ssp585       0.92291         0.88610        0.80223   
                  litter  ssp126       0.31483         0.30200        0.33962   
                          ssp585       0.23194         0.19387        0.27392   
                  shannon ssp126       0.45408         0.37382        0.56794   
                          ssp585       0.37615         0.29381        0.40493   

                                  spbau_stop  tobd_manbau  tobd_shortrot  \
Lon      Lat      ES      ssp                                              
-8.75000 42.75000 cpool   ssp126     0.00013      0.00000        0.00000   
                          ssp585     0.00010      0.00000        0.00000   
                  harvest ssp126     0.00000      0.00000        0.00000   
                          ssp585     0.00000      0.00001        0.00002   
                  litter  ssp126     0.00000      0.00000        0.00000   
...                                      ...          ...            ...   
24.75000 61.75000 harvest ssp585     0.00000      0.91402        1.00000   
                  litter  ssp126     0.00000      0.79476        0.78521   
                          ssp585     0.00273      0.92152        0.75166   
                  shannon ssp126     1.00000      0.11320        0.00000   
                          ssp585     1.00000      0.06049        0.00000   

                                  tobd_longrot  tone_manbau  tone_shortrot  \
Lon      Lat      ES      ssp                                                
-8.75000 42.75000 cpool   ssp126       0.00000      0.93456        0.87456   
                          ssp585       0.00000      0.93011        0.86648   
                  harvest ssp126       0.00000      0.98398        1.00000   
                          ssp585       0.00002      0.90788        1.00000   
                  litter  ssp126       0.00000      0.99763        1.00000   
...                                        ...          ...            ...   
24.75000 61.75000 harvest ssp585       0.77407      0.83768        0.81569   
                  litter  ssp126       1.00000      0.19043        0.17552   
                          ssp585       1.00000      0.13413        0.00000   
                  shannon ssp126       0.15378      0.53499        0.39601   
                          ssp585       0.15894      0.34035        0.25628   

                                  tone_longrot  
Lon      Lat      ES      ssp                   
-8.75000 42.75000 cpool   ssp126       1.00000  
                          ssp585       1.00000  
                  harvest ssp126       0.84611  
                          ssp585       0.79122  
                  litter  ssp126       0.95852  
...                                        ...  
24.75000 61.75000 harvest ssp585       0.76311  
                  litter  ssp126       0.22192  
                          ssp585       0.07868  
                  shannon ssp126       0.59375  
                          ssp585       0.44498  

[2192 rows x 10 columns]

In [234]:
importlib.reload(opt)

# TODO weights should be a dict.
optimized_data_original = data_for_optimizer.groupby(['Lon', 'Lat'], group_keys=False).apply(lambda gc: opt.optimize_gridcell(gc.reset_index(), location=gc.name, management_options=LPJ_MANAGEMENT_OPTIONS, climate_scenarios=SSPs, scenario_columnname='ssp', es_columnname='ES', es=ES, es_weights=[0.0, 0.0, 1.0, 0.0]))

In [235]:
optimization_plots.deck_plot(optimized_data_original.reset_index().rename(columns={'Lon': 'lon', 'Lat': 'lat'}), management_forms=LPJ_MANAGEMENT_OPTIONS, management_colors=MANAGEMENT_COLORS, radius_km=15)

{
  "initialViewState": {
    "latitude": 53.43978102189781,
    "longitude": 9.985401459854014,
    "zoom": 8
  },
  "layers": [
    {
      "@@type": "PolygonLayer",
      "autoHighlight": true,
      "data": [
        {
          "color": [
            0,
            0,
            255,
            220
          ],
          "name": "Portfolio",
          "polygon": [
            [
              -8.75,
              42.75
            ],
            [
              -8.565973045359886,
              42.75
            ],
            [
              -8.565973045359886,
              42.75
            ],
            [
              -8.75,
              42.75
            ]
          ],
          "slice": "spbau_manbau",
          "value": 0.0
        },
        {
          "color": [
            0,
            0,
            178,
            220
          ],
          "name": "Portfolio",
          "polygon": [
            [
              -8.75,
              42.75
            ],
            [
              -8.565973045359886,
              42.75
            ],
            [
              -8.565973045359886,
              42.75
            ],
            [
              -8.75,
              42.75
            ]
          ],
          "slice": "spbau_shortrot",
          "value": 0.0
        },
        {
          "color": [
            127,
            127,
            255,
            220
          ],
          "name": "Portfolio",
          "polygon": [
            [
              -8.75,
              42.75
            ],
            [
              -8.565973045359886,
              42.75
            ],
            [
              -8.565973045359886,
              42.75
            ],
            [
              -8.75,
              42.75
            ]
          ],
          "slice": "spbau_longrot",
          "value": 0.0
        },
        {
          "color": [
            128,
            0,
            128,
            220
          ],
          "name": "Portfolio",
          "polygon": [
            [
              -8.75,
              42.75
            ],
            [
              -8.565973045359886,
              42.75
            ],
            [
              -8.565973045359886,
              42.75
            ],
            [
              -8.75,
              42.75
            ]
          ],
          "slice": "spbau_stop",
          "value": 0.0
        },
        {
          "color": [
            0,
            128,
            0,
            220
          ],
          "name": "Portfolio",
          "polygon": [
            [
              -8.75,
              42.75
            ],
            [
              -8.565973045359886,
              42.75
            ],
            [
              -8.565973045359886,
              42.75
            ],
            [
              -8.75,
              42.75
            ]
          ],
          "slice": "tobd_manbau",
          "value": 0.0
        },
        {
          "color": [
            0,
            89,
            0,
            220
          ],
          "name": "Portfolio",
          "polygon": [
            [
              -8.75,
              42.75
            ],
            [
              -8.565973045359886,
              42.75
            ],
            [
              -8.565973045359886,
              42.75
            ],
            [
              -8.75,
              42.75
            ]
          ],
          "slice": "tobd_shortrot",
          "value": 0.0
        },
        {
          "color": [
            64,
            128,
            64,
            220
          ],
          "name": "Portfolio",
          "polygon": [
            [
              -8.75,
              42.75
            ],
            [
              -8.565973045359886,
              42.75
            ],
            [
              -8.565973045359886,
              42.75
            ],
            [
              -8.75,
              42.75
            ]
          ],
          "s